In [ ]:
import polars as pl
df = pl.scan_parquet("../data/processed/parquet/20250101.parquet").collect()

In [25]:
df.head()

timestamp,fecha,hora,codigo_linea,nombre_linea,codigo_estacion,nombre_estacion,entradas,salidas,total
datetime[μs],date,time,i64,str,i64,str,i64,i64,i64
2025-01-01 00:00:00,2025-01-01,00:00:00,33,"""Zona B AutoNorte""",2000,"""Cabecera Autopista Norte""",0,17,17
2025-01-01 00:00:00,2025-01-01,00:00:00,33,"""Zona B AutoNorte""",2001,"""Centro Comercial Santa Fe""",0,1,1
2025-01-01 00:00:00,2025-01-01,00:00:00,33,"""Zona B AutoNorte""",2101,"""Toberin - Foundever""",0,3,3
2025-01-01 00:00:00,2025-01-01,00:00:00,33,"""Zona B AutoNorte""",2102,"""Calle 161""",0,1,1
2025-01-01 00:00:00,2025-01-01,00:00:00,33,"""Zona B AutoNorte""",2103,"""Mazurén""",0,1,1


In [40]:
df = (
    df.with_columns(
        pl.when(pl.col("nombre_linea") == "Line for Intermedium Gate")
        .then(pl.lit("Zona G NQS Sur"))
        .otherwise(pl.col("nombre_linea"))
        .alias("nombre_linea")
    )
    .with_columns(
        pl.when(pl.col("nombre_linea") == "Zona F Calle 13")
        .then(pl.lit("Zona F Av. Américas"))
        .otherwise(pl.col("nombre_linea"))
        .alias("nombre_linea")
    )
)

In [41]:
list(df['nombre_linea'].unique().sort())

['Zona A Caracas',
 'Zona B AutoNorte',
 'Zona C Av. Suba',
 'Zona D Calle 80',
 'Zona E NQS Central',
 'Zona F Av. Américas',
 'Zona G NQS Sur',
 'Zona H Caracas Sur',
 'Zona J Eje Ambiental',
 'Zona K Calle 26',
 'Zona L Carrera 10',
 'Zona T Ciudad Bolívar']

In [42]:
list(df.filter(pl.col('nombre_linea') == 'Zona B AutoNorte')['nombre_estacion'].unique().sort())

['Alcalá – Colegio S. Tomás Dominicos',
 'Cabecera Autopista Norte',
 'Calle 100 - Marketmedios',
 'Calle 106',
 'Calle 127',
 'Calle 142',
 'Calle 161',
 'Calle 85',
 'Centro Comercial Santa Fe',
 'Heroes - Gel hada',
 'Mazurén',
 'Pepe Sierra',
 'Prado',
 'Terminal',
 'Toberin - Foundever',
 'Virrey']

In [69]:
list(df.filter(pl.col('nombre_linea') == 'Zona G NQS Sur')['nombre_estacion'].unique())

['SANTA ISABEL',
 'VENECIA',
 'Ampliacion San Mateo',
 'LEON XIII',
 'MADELENA',
 'SEVILLANA',
 'PERDOMO',
 'COMUNEROS',
 'NQS - Calle 38A Sur',
 'General Santander',
 'TERREROS',
 'NQS - Calle 30 Sur',
 'Portal Sur JFK Coop. Financiera',
 'SAN MATEO - C.C. UNISUR',
 'Bosa',
 'DESPENSA',
 'ALQUERIA',
 ' Intermedias San Mateo']

On the "Zona G NQS Sur" line there is a station called "San Mateo - C.C. Unisur" which has different places in these datasets ("Intermedias San Mateo", "Ampliación San Mateo"). We need to merge them to the main name and update the `dataset.py` file.

In [ ]:
san_mateo_variantes = df.filter(
    pl.col('nombre_estacion').str.contains(r'.*San Mateo$')
)

san_mateo_unisur = df.filter(
    pl.col('nombre_estacion') == 'SAN MATEO - C.C. UNISUR'
)

san_mateo_variantes_agg = san_mateo_variantes.group_by('timestamp').agg([
    pl.col('entradas').sum(),
    pl.col('salidas').sum(),
    pl.col('total').sum(),
])

resultado = san_mateo_unisur.join(
    san_mateo_variantes_agg, on='timestamp', how='left', suffix='_var'
).with_columns([
    (pl.col('entradas') + pl.col('entradas_var')).alias('entradas'),
    (pl.col('salidas') + pl.col('salidas_var')).alias('salidas'),
    (pl.col('total') + pl.col('total_var')).alias('total'),
]).drop(['entradas_var', 'salidas_var', 'total_var'])

df_clean = df.filter(
    ~pl.col('nombre_estacion').str.contains(r'.*San Mateo$') &
    (pl.col('nombre_estacion') != 'SAN MATEO - C.C. UNISUR')
).vstack(resultado)

And we notice that we must normalize "nombre_linea" and "nombre_estacion" columns and replace "Cabecera" to "Portal"

In [45]:
df_clean = (
    df_clean.with_columns([
        pl.col('nombre_linea', 'nombre_estacion')
            .str.to_titlecase()
            .str.strip_chars()
            .str.replace_all(r'[áàâä]', 'a')
            .str.replace_all(r'[éèêë]', 'e')
            .str.replace_all(r'[íìîï]', 'i')
            .str.replace_all(r'[óòôö]', 'o')
            .str.replace_all(r'[úùûü]', 'u')
            .str.replace_all(r'[ñ]', 'n')
            .str.replace_all(r'[Á]', 'A')
            .str.replace_all(r'[É]', 'E')
            .str.replace_all(r'[Í]', 'I')
            .str.replace_all(r'[Ó]', 'O')
            .str.replace_all(r'[Ú]', 'U')
            .str.replace_all(r'[Ñ]', 'N')
    ])
    .with_columns(
        pl.col("nombre_estacion").str.replace(r"^Cabecera", "Portal")
    )
)